# SmartRoute-OSM: Algoritmo A* (A-Estrela)

Neste terceiro notebook, carregaremos a malha viária e implementaremos o algoritmo **A*** utilizando a distância em linha reta (**Fórmula de Haversine**) como heurística geográfica para acelerar a busca do menor caminho.

In [7]:
import os
import time
import math
import heapq
import osmnx as ox
import pandas as pd

# Carregar o grafo pré-processado
data_path = "../data/quixada_drive.graphml"

if os.path.exists(data_path):
    G = ox.load_graphml(data_path)
    print(f"Grafo carregado com sucesso! Nós: {len(G.nodes)}, Arestas: {len(G.edges)}")
else:
    raise FileNotFoundError("Arquivo 'quixada_drive.graphml' não encontrado. Execute o Notebook 01 primeiro.")

Grafo carregado com sucesso! Nós: 3647, Arestas: 9762


## 1. Definição do Par Origem-Destino
Utilizamos os mesmos pontos do notebook anterior para garantir um comparativo justo entre os algoritmos.

In [8]:
# Mapear coordenadas fixas para os nós
origem_coords = (-4.9685, -39.0161)
destino_coords = (-4.9780, -39.0050)

origem_node = ox.distance.nearest_nodes(G, X=origem_coords[1], Y=origem_coords[0])
destino_node = ox.distance.nearest_nodes(G, X=destino_coords[1], Y=destino_coords[0])

print(f"ID Nó Origem: {origem_node}")
print(f"ID Nó Destino: {destino_node}")

ID Nó Origem: 252615233
ID Nó Destino: 4829461035


## 2. Função Heurística de Haversine
A heurística estima a distância geométrica em metros entre qualquer nó $n$ do grafo e o nó de destino, levando em consideração a curvatura da Terra.

In [9]:
def haversine_heuristic(node_a, node_b, graph):
    """
    Calcula a distância em metros em linha reta entre dois nós (Haversine).
    """
    y1, x1 = float(graph.nodes[node_a]['y']), float(graph.nodes[node_a]['x'])
    y2, x2 = float(graph.nodes[node_b]['y']), float(graph.nodes[node_b]['x'])
    
    R = 6371000  # Raio médio da Terra em metros
    phi1, phi2 = math.radians(y1), math.radians(y2)
    dphi = math.radians(y2 - y1)
    dlambda = math.radians(x2 - x1)

    a = math.sin(dphi / 2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    
    return R * c

## 3. Implementação do Algoritmo A*
O algoritmo prioriza a expansão de nós com base na função $f(n) = g(n) + h(n)$, onde $g(n)$ é o custo acumulado até o nó e $h(n)$ é a estimativa heurística até o destino.

In [10]:
def astar_routing(graph, start_node, target_node, weight_attribute='length'):
    """
    Calcula o menor caminho usando o algoritmo A*.
    """
    g_score = {node: float('inf') for node in graph.nodes}
    g_score[start_node] = 0
    
    f_score = {node: float('inf') for node in graph.nodes}
    f_score[start_node] = haversine_heuristic(start_node, target_node, graph)
    
    predecessors = {node: None for node in graph.nodes}
    priority_queue = [(f_score[start_node], start_node)]
    nodes_visited = 0

    while priority_queue:
        _, current_node = heapq.heappop(priority_queue)
        nodes_visited += 1

        if current_node == target_node:
            break

        for neighbor in graph.neighbors(current_node):
            edge_data = graph.get_edge_data(current_node, neighbor)[0]
            edge_weight = float(edge_data.get(weight_attribute, 1))
            
            tentative_g = g_score[current_node] + edge_weight

            if tentative_g < g_score[neighbor]:
                predecessors[neighbor] = current_node
                g_score[neighbor] = tentative_g
                h_cost = haversine_heuristic(neighbor, target_node, graph)
                f_score[neighbor] = tentative_g + h_cost
                heapq.heappush(priority_queue, (f_score[neighbor], neighbor))

    # Reconstrução do caminho
    path = []
    curr = target_node
    while curr is not None:
        path.append(curr)
        curr = predecessors[curr]
    path.reverse()

    return path, g_score[target_node], nodes_visited

## 4. Execução e Medição de Desempenho

In [11]:
start_time = time.time()
astar_path, astar_dist, astar_visited = astar_routing(G, origem_node, destino_node, weight_attribute='length')
execution_time_ms = (time.time() - start_time) * 1000

print(f"=== RESULTADOS A* ===")
print(f"Distância Total: {astar_dist:.2f} metros")
print(f"Nós Visitados: {astar_visited}")
print(f"Tempo de Execução: {execution_time_ms:.2f} ms")

=== RESULTADOS A* ===
Distância Total: 1890.27 metros
Nós Visitados: 81
Tempo de Execução: 5.64 ms


## 5. Salvando Resultados Intermediários

In [12]:
metrics_data = {
    'algorithm': ['A* (Haversine)'],
    'distance_m': [astar_dist],
    'visited_nodes': [astar_visited],
    'execution_time_ms': [execution_time_ms],
    'origem_node': [origem_node],
    'destino_node': [destino_node]
}

df_astar = pd.DataFrame(metrics_data)
df_astar.to_csv("../data/astar_metrics.csv", index=False)

# Salvar o caminho para reusar no mapa
import json
with open("../data/astar_path.json", "w") as f:
    json.dump(astar_path, f)

print("Métricas e caminho salvos na pasta '../data/' com sucesso!")

Métricas e caminho salvos na pasta '../data/' com sucesso!
